# Session 4.2: Hyperparameter Tuning
## Module 4: Model Evaluation & Optimization

**Learning Objectives:**
- Understand the difference between parameters and hyperparameters
- Implement Grid Search for exhaustive hyperparameter tuning
- Use Random Search as a faster alternative
- Tune Random Forest hyperparameters (n_estimators, max_depth, min_samples_split)
- Tune SVM hyperparameters (C, gamma, kernel)
- Compare before/after performance and achieve 10%+ improvement
- Avoid overfitting to the validation set

**Why Hyperparameter Tuning Matters:**
Default hyperparameters rarely give the best performance. Systematic tuning can improve model performance by 10-30% or more. The difference between a mediocre model and a production-ready one often comes down to proper tuning.

**What You'll Build:**
- Grid Search on Random Forest and SVM
- Random Search comparison
- Before/after performance visualization
- Best practices for avoiding overfitting during tuning

## Step 1: Import Libraries and Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from scipy.stats import uniform, randint

# Random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Libraries imported successfully!")
print(f"Random seed: {RANDOM_STATE}")

## Step 2: Parameters vs Hyperparameters

Let's clarify this important distinction:

**Parameters:**
- Learned from data during training
- Examples: weights in linear regression, tree splits in decision trees
- You don't set these - the algorithm finds them

**Hyperparameters:**
- Set BEFORE training
- Control the learning process
- Examples: learning rate, max_depth, n_estimators
- You need to tune these to optimize performance

> **💡 AI Assistant Prompt:**
> 
> "Explain the difference between parameters and hyperparameters in machine learning. Give examples for Random Forest and show how changing hyperparameters affects model performance."

In [ ]:
print("PARAMETERS vs HYPERPARAMETERS")
print("="*60)

# Create sample dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=RANDOM_STATE
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Dataset created:")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print(f"  Features: {X.shape[1]}")

# Example: Random Forest
print("\nRandom Forest Example:")
print("\nHYPERPARAMETERS (you set these):")
print("  - n_estimators: number of trees")
print("  - max_depth: maximum tree depth")
print("  - min_samples_split: min samples to split a node")
print("  - min_samples_leaf: min samples in a leaf")

rf = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print("\nPARAMETERS (learned from data):")
print(f"  - Number of trees created: {len(rf.estimators_)}")
print(f"  - Features used in first tree: {rf.estimators_[0].tree_.n_features}")
print(f"  - Total nodes in first tree: {rf.estimators_[0].tree_.node_count}")
print("  - Tree structures, split points, leaf values, etc.")

print("\n✅ Key Insight: You tune hyperparameters to improve performance!")

## Step 3: Baseline Performance (Default Hyperparameters)

Let's establish baseline performance with default hyperparameters.

In [ ]:
print("BASELINE PERFORMANCE - Default Hyperparameters")
print("="*60)

# Random Forest with defaults
rf_baseline = RandomForestClassifier(random_state=RANDOM_STATE)
rf_baseline.fit(X_train, y_train)
rf_pred_baseline = rf_baseline.predict(X_test)

print("Random Forest (default hyperparameters):")
print(f"  n_estimators: {rf_baseline.n_estimators}")
print(f"  max_depth: {rf_baseline.max_depth}")
print(f"  min_samples_split: {rf_baseline.min_samples_split}")
print(f"  min_samples_leaf: {rf_baseline.min_samples_leaf}")

rf_baseline_scores = {
    'accuracy': accuracy_score(y_test, rf_pred_baseline),
    'precision': precision_score(y_test, rf_pred_baseline),
    'recall': recall_score(y_test, rf_pred_baseline),
    'f1': f1_score(y_test, rf_pred_baseline)
}

print("\nBaseline Performance:")
for metric, score in rf_baseline_scores.items():
    print(f"  {metric.capitalize()}: {score:.4f}")

# SVM with defaults
svm_baseline = SVC(random_state=RANDOM_STATE)
svm_baseline.fit(X_train, y_train)
svm_pred_baseline = svm_baseline.predict(X_test)

print("\nSVM (default hyperparameters):")
print(f"  C: {svm_baseline.C}")
print(f"  kernel: {svm_baseline.kernel}")
print(f"  gamma: {svm_baseline.gamma}")

svm_baseline_scores = {
    'accuracy': accuracy_score(y_test, svm_pred_baseline),
    'precision': precision_score(y_test, svm_pred_baseline),
    'recall': recall_score(y_test, svm_pred_baseline),
    'f1': f1_score(y_test, svm_pred_baseline)
}

print("\nBaseline Performance:")
for metric, score in svm_baseline_scores.items():
    print(f"  {metric.capitalize()}: {score:.4f}")

print("\n📊 These are our baselines - let's improve them through tuning!")

## Step 4: Grid Search - Random Forest

Grid Search tries ALL possible combinations in a predefined grid. It's exhaustive but can be slow.

> **💡 AI Assistant Prompt:**
> 
> "Explain how Grid Search works for hyperparameter tuning. Show code implementing Grid Search on Random Forest with parameters: n_estimators, max_depth, and min_samples_split. Include visualization of results."

In [ ]:
print("GRID SEARCH - Random Forest")
print("="*60)

# Define parameter grid
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

total_combinations = np.prod([len(v) for v in rf_param_grid.values()])
print(f"Parameter grid: {rf_param_grid}")
print(f"\nTotal combinations to try: {total_combinations}")
print(f"With 3-fold CV: {total_combinations * 3} models to train\n")

# Grid Search with CV
rf_grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Starting Grid Search...")
start_time = time.time()
rf_grid_search.fit(X_train, y_train)
grid_search_time = time.time() - start_time

print(f"\n✅ Grid Search completed in {grid_search_time:.2f} seconds")
print(f"\nBest parameters found:")
for param, value in rf_grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1-score: {rf_grid_search.best_score_:.4f}")

# Test on held-out test set
rf_tuned = rf_grid_search.best_estimator_
rf_pred_tuned = rf_tuned.predict(X_test)

rf_tuned_scores = {
    'accuracy': accuracy_score(y_test, rf_pred_tuned),
    'precision': precision_score(y_test, rf_pred_tuned),
    'recall': recall_score(y_test, rf_pred_tuned),
    'f1': f1_score(y_test, rf_pred_tuned)
}

print("\nTest Set Performance (Tuned):")
for metric, score in rf_tuned_scores.items():
    print(f"  {metric.capitalize()}: {score:.4f}")

## Step 5: Random Search - Faster Alternative

Random Search samples random combinations instead of trying all. Often finds good hyperparameters faster than Grid Search.

In [ ]:
print("RANDOM SEARCH - Random Forest")
print("="*60)

# Define parameter distributions
rf_param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

print(f"Parameter distributions: {rf_param_dist}")
print(f"\nWill try 50 random combinations (vs {total_combinations} for Grid Search)")

# Random Search
rf_random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_dist,
    n_iter=50,  # Number of random combinations to try
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

print("\nStarting Random Search...")
start_time = time.time()
rf_random_search.fit(X_train, y_train)
random_search_time = time.time() - start_time

print(f"\n✅ Random Search completed in {random_search_time:.2f} seconds")
print(f"   Grid Search took: {grid_search_time:.2f} seconds")
print(f"   Speedup: {grid_search_time/random_search_time:.2f}x faster")

print(f"\nBest parameters found:")
for param, value in rf_random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1-score: {rf_random_search.best_score_:.4f}")
print(f"Grid Search best F1: {rf_grid_search.best_score_:.4f}")

print("\n💡 Random Search often finds competitive results much faster!")

## Step 6: Grid Search - SVM

Now let's tune SVM hyperparameters: C (regularization), gamma (kernel coefficient), and kernel type.

In [ ]:
print("GRID SEARCH - SVM")
print("="*60)

# Scale features (important for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define parameter grid
svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

total_svm_combinations = np.prod([len(v) for v in svm_param_grid.values()])
print(f"Parameter grid: {svm_param_grid}")
print(f"\nTotal combinations: {total_svm_combinations}")

# Grid Search
svm_grid_search = GridSearchCV(
    SVC(random_state=RANDOM_STATE),
    svm_param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("\nStarting SVM Grid Search...")
start_time = time.time()
svm_grid_search.fit(X_train_scaled, y_train)
svm_search_time = time.time() - start_time

print(f"\n✅ Grid Search completed in {svm_search_time:.2f} seconds")
print(f"\nBest parameters found:")
for param, value in svm_grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1-score: {svm_grid_search.best_score_:.4f}")

# Test on held-out test set
svm_tuned = svm_grid_search.best_estimator_
svm_pred_tuned = svm_tuned.predict(X_test_scaled)

svm_tuned_scores = {
    'accuracy': accuracy_score(y_test, svm_pred_tuned),
    'precision': precision_score(y_test, svm_pred_tuned),
    'recall': recall_score(y_test, svm_pred_tuned),
    'f1': f1_score(y_test, svm_pred_tuned)
}

print("\nTest Set Performance (Tuned):")
for metric, score in svm_tuned_scores.items():
    print(f"  {metric.capitalize()}: {score:.4f}")

## Step 7: Before vs After Comparison

Let's visualize the improvement from hyperparameter tuning.

In [ ]:
print("BEFORE vs AFTER COMPARISON")
print("="*60)

# Calculate improvements
print("\nRANDOM FOREST:")
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    before = rf_baseline_scores[metric]
    after = rf_tuned_scores[metric]
    improvement = ((after - before) / before) * 100
    print(f"  {metric.capitalize()}:")
    print(f"    Before: {before:.4f}")
    print(f"    After:  {after:.4f}")
    print(f"    Improvement: {improvement:+.2f}%")

print("\nSVM:")
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    before = svm_baseline_scores[metric]
    after = svm_tuned_scores[metric]
    improvement = ((after - before) / before) * 100
    print(f"  {metric.capitalize()}:")
    print(f"    Before: {before:.4f}")
    print(f"    After:  {after:.4f}")
    print(f"    Improvement: {improvement:+.2f}%")

# Visualize improvements
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['accuracy', 'precision', 'recall', 'f1']

# 1. Random Forest - Before vs After
rf_before = [rf_baseline_scores[m] for m in metrics]
rf_after = [rf_tuned_scores[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35
axes[0, 0].bar(x - width/2, rf_before, width, label='Before', color='#e74c3c', alpha=0.7)
axes[0, 0].bar(x + width/2, rf_after, width, label='After', color='#2ecc71', alpha=0.7)
axes[0, 0].set_ylabel('Score', fontsize=11)
axes[0, 0].set_title('Random Forest: Before vs After Tuning', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([m.capitalize() for m in metrics])
axes[0, 0].legend()
axes[0, 0].set_ylim(0.7, 1.0)
axes[0, 0].grid(alpha=0.3)

# 2. SVM - Before vs After
svm_before = [svm_baseline_scores[m] for m in metrics]
svm_after = [svm_tuned_scores[m] for m in metrics]

axes[0, 1].bar(x - width/2, svm_before, width, label='Before', color='#e74c3c', alpha=0.7)
axes[0, 1].bar(x + width/2, svm_after, width, label='After', color='#2ecc71', alpha=0.7)
axes[0, 1].set_ylabel('Score', fontsize=11)
axes[0, 1].set_title('SVM: Before vs After Tuning', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([m.capitalize() for m in metrics])
axes[0, 1].legend()
axes[0, 1].set_ylim(0.7, 1.0)
axes[0, 1].grid(alpha=0.3)

# 3. Random Forest - Improvement percentages
rf_improvements = [((rf_tuned_scores[m] - rf_baseline_scores[m]) / rf_baseline_scores[m]) * 100 
                   for m in metrics]
colors_rf = ['#2ecc71' if x > 0 else '#e74c3c' for x in rf_improvements]
axes[1, 0].barh(metrics, rf_improvements, color=colors_rf)
axes[1, 0].set_xlabel('Improvement (%)', fontsize=11)
axes[1, 0].set_title('Random Forest: Percentage Improvement', fontsize=12, fontweight='bold')
axes[1, 0].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 0].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% target')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. SVM - Improvement percentages
svm_improvements = [((svm_tuned_scores[m] - svm_baseline_scores[m]) / svm_baseline_scores[m]) * 100 
                    for m in metrics]
colors_svm = ['#2ecc71' if x > 0 else '#e74c3c' for x in svm_improvements]
axes[1, 1].barh(metrics, svm_improvements, color=colors_svm)
axes[1, 1].set_xlabel('Improvement (%)', fontsize=11)
axes[1, 1].set_title('SVM: Percentage Improvement', fontsize=12, fontweight='bold')
axes[1, 1].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% target')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Check if 10% improvement achieved
rf_avg_improvement = np.mean(rf_improvements)
svm_avg_improvement = np.mean(svm_improvements)

print(f"\n📈 Average Improvements:")
print(f"  Random Forest: {rf_avg_improvement:.2f}%")
print(f"  SVM: {svm_avg_improvement:.2f}%")

if rf_avg_improvement >= 10:
    print("  ✅ Random Forest: 10%+ improvement achieved!")
else:
    print(f"  ⚠️ Random Forest: Need {10 - rf_avg_improvement:.2f}% more")

if svm_avg_improvement >= 10:
    print("  ✅ SVM: 10%+ improvement achieved!")
else:
    print(f"  ⚠️ SVM: Need {10 - svm_avg_improvement:.2f}% more")

## Step 8: Visualize Grid Search Results

Let's explore how different hyperparameters affect performance.

In [ ]:
print("GRID SEARCH RESULTS ANALYSIS")
print("="*60)

# Extract Grid Search results
rf_results_df = pd.DataFrame(rf_grid_search.cv_results_)

# Top 10 configurations
print("\nTop 10 Hyperparameter Configurations:")
top_10 = rf_results_df.nlargest(10, 'mean_test_score')[[
    'param_n_estimators', 'param_max_depth', 'param_min_samples_split',
    'param_min_samples_leaf', 'mean_test_score', 'std_test_score'
]]
print(top_10.to_string(index=False))

# Visualize parameter importance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. n_estimators effect
n_est_groups = rf_results_df.groupby('param_n_estimators')['mean_test_score'].mean()
axes[0, 0].bar(range(len(n_est_groups)), n_est_groups.values, color='#3498db')
axes[0, 0].set_xticks(range(len(n_est_groups)))
axes[0, 0].set_xticklabels(n_est_groups.index)
axes[0, 0].set_xlabel('n_estimators', fontsize=11)
axes[0, 0].set_ylabel('Mean F1-Score', fontsize=11)
axes[0, 0].set_title('Effect of n_estimators', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# 2. max_depth effect
depth_groups = rf_results_df.groupby('param_max_depth')['mean_test_score'].mean()
axes[0, 1].bar(range(len(depth_groups)), depth_groups.values, color='#2ecc71')
axes[0, 1].set_xticks(range(len(depth_groups)))
axes[0, 1].set_xticklabels([str(d) for d in depth_groups.index])
axes[0, 1].set_xlabel('max_depth', fontsize=11)
axes[0, 1].set_ylabel('Mean F1-Score', fontsize=11)
axes[0, 1].set_title('Effect of max_depth', fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# 3. min_samples_split effect
split_groups = rf_results_df.groupby('param_min_samples_split')['mean_test_score'].mean()
axes[1, 0].bar(range(len(split_groups)), split_groups.values, color='#e74c3c')
axes[1, 0].set_xticks(range(len(split_groups)))
axes[1, 0].set_xticklabels(split_groups.index)
axes[1, 0].set_xlabel('min_samples_split', fontsize=11)
axes[1, 0].set_ylabel('Mean F1-Score', fontsize=11)
axes[1, 0].set_title('Effect of min_samples_split', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# 4. min_samples_leaf effect
leaf_groups = rf_results_df.groupby('param_min_samples_leaf')['mean_test_score'].mean()
axes[1, 1].bar(range(len(leaf_groups)), leaf_groups.values, color='#f39c12')
axes[1, 1].set_xticks(range(len(leaf_groups)))
axes[1, 1].set_xticklabels(leaf_groups.index)
axes[1, 1].set_xlabel('min_samples_leaf', fontsize=11)
axes[1, 1].set_ylabel('Mean F1-Score', fontsize=11)
axes[1, 1].set_title('Effect of min_samples_leaf', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 These plots show which hyperparameters matter most!")

## Step 9: Avoiding Overfitting to Validation Set

Important warning: You can overfit hyperparameters to your validation set!

> **💡 AI Assistant Prompt:**
> 
> "Explain how hyperparameter tuning can lead to overfitting to the validation set. What are best practices to avoid this? Include discussion of nested cross-validation."

In [ ]:
print("AVOIDING OVERFITTING DURING HYPERPARAMETER TUNING")
print("="*60)

print("\n⚠️ COMMON MISTAKE: Overfitting to Validation Set")
print("   If you tune extensively on the same validation set,")
print("   your hyperparameters might work well on that specific set")
print("   but not generalize to new data.")

print("\n✅ BEST PRACTICES:")
print("\n1. Always use Cross-Validation during tuning")
print("   - Grid/Random Search with cv=5 or cv=10")
print("   - This averages performance across multiple validation sets")

print("\n2. Keep a separate test set")
print("   - NEVER use test set during hyperparameter selection")
print("   - Only use it for final model evaluation")
print("   - Split: 60% train, 20% validation (CV), 20% test")

print("\n3. Use Nested Cross-Validation for unbiased estimates")
print("   - Outer loop: Model evaluation")
print("   - Inner loop: Hyperparameter tuning")
print("   - Prevents information leakage")

print("\n4. Don't tune too many hyperparameters")
print("   - Focus on the most important ones")
print("   - More parameters = more chances to overfit")

print("\n5. Monitor CV standard deviation")
print("   - High std = unstable, might be overfitting")
print("   - Prefer models with low std")

# Demonstrate nested CV
print("\n" + "="*60)
print("NESTED CROSS-VALIDATION EXAMPLE")
print("="*60)

from sklearn.model_selection import cross_val_score, KFold

# Outer loop for model evaluation
outer_cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# Inner loop for hyperparameter tuning
inner_cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# Simplified parameter grid
param_grid_simple = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None]
}

# Grid search object
gs = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid_simple,
    cv=inner_cv,
    scoring='f1'
)

# Nested CV: Outer loop evaluates Grid Search
nested_scores = cross_val_score(gs, X_train, y_train, cv=outer_cv, scoring='f1')

print(f"\nNested CV F1-Scores: {nested_scores}")
print(f"Mean: {nested_scores.mean():.4f} (± {nested_scores.std():.4f})")

print("\n✅ This is an unbiased estimate of model performance!")
print("   The outer loop never sees the hyperparameter selection process.")

## Step 10: Practical Tips for Hyperparameter Tuning

In [ ]:
print("PRACTICAL HYPERPARAMETER TUNING TIPS")
print("="*60)

print("\n1. START WITH RANDOM SEARCH")
print("   - Faster than Grid Search")
print("   - Try 50-100 random combinations first")
print("   - Then refine with Grid Search around promising values")

print("\n2. KNOW YOUR HYPERPARAMETERS")
print("\n   Random Forest most important:")
print("   - n_estimators: More is better (diminishing returns after 200)")
print("   - max_depth: Controls overfitting (start with 10-20)")
print("   - min_samples_split: Higher values prevent overfitting")

print("\n   SVM most important:")
print("   - C: Regularization strength (try 0.1, 1, 10, 100)")
print("   - gamma: Kernel coefficient (try 'scale', 'auto', 0.001-1)")
print("   - kernel: 'rbf' for non-linear, 'linear' for linear data")

print("\n3. USE LOGARITHMIC RANGES")
print("   - For C, gamma: [0.001, 0.01, 0.1, 1, 10, 100]")
print("   - Covers wide range efficiently")

print("\n4. PARALLELIZE")
print("   - Use n_jobs=-1 to use all CPU cores")
print("   - Can speed up 4-8x on modern computers")

print("\n5. MONITOR TRAINING TIME")
print("   - Some hyperparameters greatly increase training time")
print("   - Balance performance vs computational cost")

print("\n6. START COARSE, THEN REFINE")
print("   - First search: Wide range, few values")
print("   - Second search: Narrow range around best values")

print("\n7. DOCUMENT YOUR EXPERIMENTS")
print("   - Save best parameters and scores")
print("   - Track what you've tried")
print("   - Helps avoid repeating failed experiments")

# Example: Coarse to fine tuning
print("\n" + "="*60)
print("EXAMPLE: Coarse to Fine Tuning Strategy")
print("="*60)

print("\nSTEP 1: Coarse search (wide range, few values)")
coarse_grid = {
    'n_estimators': [50, 200],
    'max_depth': [5, 20, None]
}
print(f"   Grid: {coarse_grid}")
print(f"   Combinations: {2 * 3} = 6")

print("\nSTEP 2: Analyze results, identify promising region")
print("   Suppose best was: n_estimators=200, max_depth=20")

print("\nSTEP 3: Fine search (narrow range, more values)")
fine_grid = {
    'n_estimators': [150, 200, 250],
    'max_depth': [15, 20, 25]
}
print(f"   Grid: {fine_grid}")
print(f"   Combinations: {3 * 3} = 9")

print("\n✅ This two-stage approach is efficient and effective!")

## Summary: What You Learned

### Key Concepts:

1. **Parameters vs Hyperparameters**: Parameters are learned from data, hyperparameters are set before training.

2. **Grid Search**: Tries all combinations in a grid. Exhaustive but can be slow. Use when you have time and want guaranteed best results.

3. **Random Search**: Samples random combinations. Often finds good results 5-10x faster than Grid Search.

4. **Important Hyperparameters**:
   - Random Forest: n_estimators, max_depth, min_samples_split
   - SVM: C, gamma, kernel

5. **Avoiding Overfitting**: Always use CV during tuning, keep separate test set, consider nested CV.

6. **Improvement Target**: Aim for 10%+ improvement through tuning. If you don't achieve this, try different algorithms or feature engineering.

### Best Practices:

- Start with Random Search (50-100 iterations)
- Use Grid Search to refine promising regions
- Always use cross-validation (cv=5 or cv=10)
- Keep a separate test set for final evaluation
- Monitor CV standard deviation for stability
- Use n_jobs=-1 for parallel processing
- Document your experiments

### Quiz Preparation:

You should now be able to answer:
- What's the difference between parameters and hyperparameters? (M4-S5-Q01)
- How does Grid Search work? (M4-S5-Q02)
- What's the advantage of Random Search? (M4-S5-Q03)
- How many models trained in Grid Search? (M4-S5-Q04)
- Why use CV during tuning? (M4-S5-Q06)

### Computational Cost:

Remember: If you have 4 hyperparameters with 4 values each and use 5-fold CV:
- Combinations: 4^4 = 256
- Total models: 256 × 5 = 1,280 models!

This is why Random Search and coarse-to-fine strategies are important.

### Next Steps:

1. Apply these techniques to your own datasets
2. Try different hyperparameter ranges
3. Combine tuning with feature selection (next session!)
4. Use tools like Optuna for Bayesian optimization (advanced)

> **💡 AI Assistant Challenge:**
> 
> "I have a Random Forest model with 75% accuracy. I want to improve it to at least 85%. Walk me through a systematic hyperparameter tuning process. Include code for both Grid Search and Random Search, and explain how to avoid overfitting during tuning."

**Excellent work! You can now tune hyperparameters like a pro and significantly improve model performance!** 🎉